# RGM Copilot — Phase 1: GenAI Layer (Function Calling + RAG)

**Goal:** Let a non-technical stakeholder ask plain-English business questions and get answers
grounded in the *actual* model outputs from notebooks 2 and 3 — not hallucinated numbers.

**Architecture:**
- **LLM:** Google Gemini (`gemini-2.5-flash`, free tier) via the official `google-genai` SDK
- **Function calling:** the model can call real Python functions that query the saved forecast
  and stockout-risk lookup tables — so questions like *"what's the forecast for X"* get a real,
  computed answer, not a guess
- **RAG (Retrieval-Augmented Generation):** a small knowledge base of business-context notes,
  each one **derived directly from the actual computed statistics** in this project (channel
  stockout rates, promo uplift, top-risk SKUs, etc.) — not invented text. Retrieval uses TF-IDF +
  cosine similarity rather than calling an embedding API for every note, which keeps this
  comfortably inside the Gemini free tier's daily request limit. (In a production system, this
  would typically use a proper embedding model + vector database — the retrieval *pattern* is
  identical, just swapped for a lighter-weight implementation here.)

**Note on execution:** the data/function/retrieval cells below were tested and run already.
The live Gemini chat cells need *your* API key to run — you'll be running those for the first time.

## 1. Setup

Install the SDK if you haven't already: `pip install google-genai` (run in Anaconda Prompt if needed, same as we did for xgboost).

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import getpass
import os

from google import genai
from google.genai import types

## 2. Enter your Gemini API key

This uses `getpass` so the key is typed securely and **never saved into this notebook file** —
safe to commit this notebook to GitHub afterwards without leaking your key.

In [2]:
os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
client = genai.Client()
print("Client ready.")

Paste your Gemini API key:  ········


Client ready.


## 3. Load the saved model outputs from notebooks 2 and 3

In [3]:
df = pd.read_csv("../data/rgm_copilot_dataset.csv", parse_dates=["date"])
forecast_lookup = pd.read_csv("../models/forecast_lookup.csv", parse_dates=["date"])
risk_lookup = pd.read_csv("../models/stockout_risk_lookup.csv", parse_dates=["date"])

print("Dataset:", df.shape)
print("Forecast lookup:", forecast_lookup.shape)
print("Risk lookup:", risk_lookup.shape)

Dataset: (116800, 12)
Forecast lookup: (29280, 6)
Risk lookup: (29280, 6)


## 4. Function-calling tools

These are plain Python functions with type hints and docstrings — the `google-genai` SDK
automatically converts them into a schema the model can call. When the model decides a question
needs real data, it calls one of these and gets the actual computed answer back.

In [4]:
def get_demand_forecast(region: str, channel: str, sku: str) -> str:
    """Get the demand forecast for a specific region, channel, and SKU combination.

    Args:
        region: The region name, e.g. 'Greater Cairo'.
        channel: The sales channel, e.g. 'Modern Trade', 'HoReCa', 'E-Commerce', 'Traditional Trade'.
        sku: The product SKU name, e.g. 'Cola Regular 330ml'.
    """
    sub = forecast_lookup[
        (forecast_lookup["region"].str.lower() == region.lower()) &
        (forecast_lookup["channel"].str.lower() == channel.lower()) &
        (forecast_lookup["sku"].str.lower() == sku.lower())
    ]
    if sub.empty:
        return f"No forecast data found for {sku} in {region}, {channel}."
    latest = sub.sort_values("date").iloc[-1]
    avg_recent = sub.sort_values("date").tail(14)["forecast"].mean()
    return (
        f"Latest forecast for {sku} in {region} ({channel}) on {latest['date'].date()}: "
        f"{latest['forecast']:.0f} units (actual was {latest['actual']:.0f}). "
        f"14-day average forecasted demand: {avg_recent:.0f} units/day."
    )


def get_stockout_risk(region: str, channel: str, sku: str) -> str:
    """Get the stockout risk assessment for a specific region, channel, and SKU combination.

    Args:
        region: The region name, e.g. 'Greater Cairo'.
        channel: The sales channel, e.g. 'Modern Trade', 'HoReCa', 'E-Commerce', 'Traditional Trade'.
        sku: The product SKU name, e.g. 'Cola Regular 330ml'.
    """
    sub = risk_lookup[
        (risk_lookup["region"].str.lower() == region.lower()) &
        (risk_lookup["channel"].str.lower() == channel.lower()) &
        (risk_lookup["sku"].str.lower() == sku.lower())
    ]
    if sub.empty:
        return f"No stockout risk data found for {sku} in {region}, {channel}."
    avg_risk = sub["stockout_risk_score"].mean()
    max_risk = sub["stockout_risk_score"].max()
    actual_stockouts = sub["stockout_actual"].sum()
    return (
        f"Stockout risk for {sku} in {region} ({channel}): "
        f"average risk score {avg_risk:.1%}, peak risk {max_risk:.1%}. "
        f"{int(actual_stockouts)} actual stockout days observed in the test period."
    )


def get_top_risk_items(n: int = 5) -> str:
    """Get the top N region/channel/SKU combinations with the highest average stockout risk.

    Args:
        n: How many top items to return. Defaults to 5.
    """
    top = (risk_lookup.groupby(["region", "channel", "sku"])["stockout_risk_score"]
           .mean().sort_values(ascending=False).head(n))
    lines = [f"{i+1}. {sku} - {region} - {channel}: {score:.1%} avg risk"
             for i, ((region, channel, sku), score) in enumerate(top.items())]
    return "Top stockout-risk items:\n" + "\n".join(lines)


# quick local sanity check (no API call yet)
print(get_demand_forecast("Greater Cairo", "Modern Trade", "Cola Regular 330ml"))
print()
print(get_top_risk_items(5))

Latest forecast for Cola Regular 330ml in Greater Cairo (Modern Trade) on 2024-12-30: 73 units (actual was 64). 14-day average forecasted demand: 89 units/day.

Top stockout-risk items:
1. Sparkling Water 500ml - Upper Egypt - HoReCa: 14.8% avg risk
2. Sparkling Water 500ml - Suez Canal Zone - HoReCa: 14.0% avg risk
3. Sparkling Water 500ml - Delta Region - HoReCa: 13.1% avg risk
4. Orange Soda 330ml - Suez Canal Zone - HoReCa: 12.5% avg risk
5. Juice 1L - Alexandria & North Coast - HoReCa: 12.3% avg risk


## 5. RAG knowledge base — grounded in real computed statistics

Every note below is generated from actual numbers computed earlier in this project, not invented.
This is what makes the RAG layer trustworthy: when the assistant answers a "why" question, it's
quoting genuine analysis, not making something plausible-sounding up.

In [5]:
notes = []

# Channel-level stockout patterns
stockout_by_channel = df.groupby("channel")["stockout_flag"].mean().sort_values(ascending=False)
worst_channel = stockout_by_channel.index[0]
notes.append(
    f"{worst_channel} has the highest stockout rate of any channel, at {stockout_by_channel.iloc[0]:.1%}. "
    f"This channel tends to have spikier, less predictable demand and thinner inventory buffers compared to "
    f"{stockout_by_channel.index[-1]}, which has the lowest stockout rate at {stockout_by_channel.iloc[-1]:.1%}."
)

# SKU-level lead time patterns
sku_info = df[["sku", "lead_time_days"]].drop_duplicates().sort_values("lead_time_days", ascending=False)
longest_lead_sku = sku_info.iloc[0]
notes.append(
    f"{longest_lead_sku['sku']} has the longest replenishment lead time in the portfolio at "
    f"{longest_lead_sku['lead_time_days']:.0f} days, making it more vulnerable to stockouts when demand spikes "
    f"unexpectedly, since there is less time to react with a reorder."
)

# Promo uplift
promo_effect = df.groupby("on_promo")["units_sold"].mean()
uplift = promo_effect[True] / promo_effect[False] - 1
notes.append(
    f"Promotional activity drives an average {uplift:.0%} uplift in units sold across the portfolio. "
    f"This uplift needs to be factored into demand forecasts and inventory planning ahead of promo periods, "
    f"otherwise stockout risk increases sharply during promotions."
)

# Top average-risk combos
top_risk = (risk_lookup.groupby(["region", "channel", "sku"])["stockout_risk_score"]
            .mean().sort_values(ascending=False).head(5))
for (region, channel, sku), score in top_risk.items():
    notes.append(
        f"{sku} in {region} through the {channel} channel is among the highest stockout-risk combinations "
        f"in the model's predictions, with an average predicted risk of {score:.1%}. "
        f"This combination should be prioritized for inventory review."
    )

# Frequency of extreme-risk days (catches repeat-offender combos an average can wash out)
extreme = risk_lookup[risk_lookup["stockout_risk_score"] > 0.9]
extreme_counts = extreme.groupby(["region", "channel", "sku"]).size().sort_values(ascending=False)
if len(extreme_counts) > 0:
    (top_region, top_channel, top_sku), top_count = extreme_counts.index[0], extreme_counts.iloc[0]
    notes.append(
        f"{top_sku} in {top_region} ({top_channel}) recorded {int(top_count)} days with extreme stockout risk "
        f"(predicted risk above 90%) during the test period — more than any other combination. "
        f"This pattern repeats across multiple {top_channel} locations for {top_sku}, suggesting a "
        f"structural issue (likely its longer lead time combined with high seasonal demand) rather than "
        f"a one-off event."
    )

# Regional revenue patterns
region_rev = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
notes.append(
    f"{region_rev.index[0]} is the highest-revenue region in the portfolio, generating "
    f"{region_rev.iloc[0]:,.0f} in total revenue over the dataset period, "
    f"reflecting its larger market size relative to other regions."
)

# Seasonality
notes.append(
    "Demand across the portfolio follows a clear summer seasonality pattern, with cold beverages "
    "(colas, sodas, sparkling water, iced tea) seeing the strongest uplift during peak summer months, "
    "increasing both sales volume and stockout risk if inventory isn't scaled up ahead of the season."
)

print(f"Built {len(notes)} grounded business-context notes.")
for n in notes:
    print("-", n)

Built 11 grounded business-context notes.
- HoReCa has the highest stockout rate of any channel, at 6.6%. This channel tends to have spikier, less predictable demand and thinner inventory buffers compared to E-Commerce, which has the lowest stockout rate at 3.4%.
- Energy Drink 250ml has the longest replenishment lead time in the portfolio at 6 days, making it more vulnerable to stockouts when demand spikes unexpectedly, since there is less time to react with a reorder.
- Promotional activity drives an average 55% uplift in units sold across the portfolio. This uplift needs to be factored into demand forecasts and inventory planning ahead of promo periods, otherwise stockout risk increases sharply during promotions.
- Sparkling Water 500ml in Upper Egypt through the HoReCa channel is among the highest stockout-risk combinations in the model's predictions, with an average predicted risk of 14.8%. This combination should be prioritized for inventory review.
- Sparkling Water 500ml in Sue

In [6]:
vectorizer = TfidfVectorizer(stop_words="english")
note_vectors = vectorizer.fit_transform(notes)

def retrieve_context(query: str, k: int = 2) -> list:
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, note_vectors)[0]
    top_idx = sims.argsort()[::-1][:k]
    return [notes[i] for i in top_idx if sims[i] > 0]

# quick local sanity check (no API call yet)
test_query = "Why does the stockout risk model flag some products more than others?"
for r in retrieve_context(test_query, k=2):
    print("-", r)

- Sparkling Water 500ml in Delta Region through the HoReCa channel is among the highest stockout-risk combinations in the model's predictions, with an average predicted risk of 13.1%. This combination should be prioritized for inventory review.
- Sparkling Water 500ml in Upper Egypt through the HoReCa channel is among the highest stockout-risk combinations in the model's predictions, with an average predicted risk of 14.8%. This combination should be prioritized for inventory review.


## 6. The RGM Copilot — combining function calling + RAG

This is the core assistant function:
1. Retrieve relevant context notes for the question (RAG)
2. Pass the question + retrieved context + the function-calling tools to Gemini
3. Gemini decides whether to call a function (for "what is X" data questions), use the retrieved
   context (for "why" questions), or both — then produces a final natural-language answer

In [7]:
SYSTEM_INSTRUCTION = '''You are RGM Copilot, an analytics assistant for an FMCG Revenue Growth
Management team. You help stakeholders understand demand forecasts and stockout risk.

Rules:
- For questions asking for specific numbers (forecasts, risk scores, top-risk items), use the
  available functions to get real data. Never make up numbers.
- For "why" questions, use the provided context notes, which are grounded in real analysis.
- Be concise and business-focused. Avoid jargon where possible.
- If you don't have enough information to answer confidently, say so rather than guessing.'''


def ask_rgm_copilot(question: str, chat=None):
    context_notes = retrieve_context(question, k=2)
    context_block = "\n".join(f"- {n}" for n in context_notes) if context_notes else "(no relevant context found)"

    augmented_input = f'''Relevant business context (from prior analysis):
{context_block}

Question: {question}'''

    if chat is None:
        chat = client.chats.create(
            model="gemini-2.5-flash",
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_INSTRUCTION,
                tools=[get_demand_forecast, get_stockout_risk, get_top_risk_items],
            ),
        )

    response = chat.send_message(augmented_input)
    return response.text, chat

## 7. Demo conversation

Three example questions, each exercising a different capability:
1. A direct data question (function calling)
2. A ranking question (function calling)
3. A "why" question (RAG)

In [8]:
answer1, chat = ask_rgm_copilot(
    "What's the demand forecast for Cola Regular 330ml in Greater Cairo, Modern Trade?"
)
print("Q: What's the demand forecast for Cola Regular 330ml in Greater Cairo, Modern Trade?\n")
print("A:", answer1)

Q: What's the demand forecast for Cola Regular 330ml in Greater Cairo, Modern Trade?

A: The demand forecast for Cola Regular 330ml in Greater Cairo (Modern Trade) for 2024-12-30 is 73 units. The 14-day average forecasted demand is 89 units/day.


In [9]:
answer2, chat = ask_rgm_copilot(
    "Which products are at the highest risk of stockout right now?", chat=chat
)
print("Q: Which products are at the highest risk of stockout right now?\n")
print("A:", answer2)

Q: Which products are at the highest risk of stockout right now?

A: The products at the highest risk of stockout are:
1. Sparkling Water 500ml in Upper Egypt (HoReCa channel) with an average risk of 14.8%.
2. Sparkling Water 500ml in Suez Canal Zone (HoReCa channel) with an average risk of 14.0%.
3. Sparkling Water 500ml in Delta Region (HoReCa channel) with an average risk of 13.1%.
4. Orange Soda 330ml in Suez Canal Zone (HoReCa channel) with an average risk of 12.5%.
5. Juice 1L in Alexandria & North Coast (HoReCa channel) with an average risk of 12.3%.


In [10]:
answer3, chat = ask_rgm_copilot(
    "Why do certain SKUs keep showing up as high stockout risk in HoReCa?", chat=chat
)
print("Q: Why do certain SKUs keep showing up as high stockout risk in HoReCa?\n")
print("A:", answer3)

Q: Why do certain SKUs keep showing up as high stockout risk in HoReCa?

A: Certain SKUs, like Juice 1L, repeatedly show up as high stockout risk in the HoReCa channel due to structural issues, specifically a combination of longer lead times and high seasonal demand for these products.


## 8. Try your own questions

Use the cell below to ask anything. This is also what the demo GIF for your README/LinkedIn
post should show — type a real question and let it answer live.

In [11]:
your_question = "Should we be worried about Energy Drink stock levels?"  # edit this
answer, chat = ask_rgm_copilot(your_question, chat=chat)
print("Q:", your_question, "\n")
print("A:", answer)

Q: Should we be worried about Energy Drink stock levels? 

A: Yes, you should be worried about Energy Drink stock levels. Energy Drink 250ml has the longest replenishment lead time at 6 days, which makes it more vulnerable to stockouts during unexpected demand spikes, as there is less time to react with reorders.
